# Customer Complaint Category Classification

**Tabular Classification Project**

## 2 · Project Overview

Classify customer complaints into 5 categories (billing, delivery, product_quality, technical_support, account) based on derived numeric features: word count, urgency indicators, response time, customer tenure, and order history. Synthetic dataset with ~5,000 rows for multiclass support ticket routing.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given numeric features derived from customer complaint tickets (word count, urgency score, response time, tenure, order count), predict the complaint category (5 classes).

## 5 · Why This Project Matters

- **Customer support routing** affects resolution time and satisfaction.
- Automatic categorization reduces manual triage effort by 50-70%.
- This teaches **multiclass classification** on derived text features.
- Fast routing improves customer retention and reduces costs.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | ~5,000 |
| Features | 8 (word_count, urgency_score, response_time_hrs, customer_tenure_months, order_count, has_attachment, priority_flag, sentiment_score) |
| Target | `category` (5 classes: 0-4, representing billing/delivery/product/technical/account) |
| Class balance | Roughly equal (~20% each) |
| Missing values | None |

## 7 · Dataset Source and License Notes

**Source**: Synthetic customer complaint categorization dataset.
**License**: Educational / synthetic.
**Note**: Simulates support ticket routing based on derived features.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "category"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Customer Complaint Category Classification


## 11 · Dataset Download or Loading

In [4]:
np.random.seed(SEED)
n = 5000

word_count = np.random.lognormal(3.5, 0.6, n).clip(10, 500).astype(int)
urgency_score = np.random.uniform(0, 10, n).round(2)
response_time = np.random.exponential(24, n).clip(0.5, 168).round(1)
tenure_months = np.random.exponential(24, n).clip(1, 120).astype(int)
order_count = np.random.poisson(8, n).clip(0, 50)
has_attachment = np.random.choice([0, 1], n, p=[0.7, 0.3])
priority_flag = np.random.choice([0, 1], n, p=[0.8, 0.2])
sentiment = np.random.normal(-0.2, 0.5, n).clip(-1, 1).round(3)

# Category based on feature combinations
base_score = np.column_stack([
    0.01 * word_count + 0.3 * urgency_score - 0.5 * sentiment,  # billing
    0.005 * response_time + 0.1 * order_count + np.random.normal(0, 0.5, n),  # delivery
    -0.3 * sentiment + 0.2 * has_attachment + np.random.normal(0, 0.5, n),  # product quality
    0.2 * priority_flag + 0.15 * urgency_score + np.random.normal(0, 0.5, n),  # technical
    0.01 * tenure_months - 0.1 * order_count + np.random.normal(0, 0.5, n),  # account
])
category = base_score.argmax(axis=1)

df = pd.DataFrame({
    'word_count': word_count, 'urgency_score': urgency_score,
    'response_time_hrs': response_time, 'customer_tenure_months': tenure_months,
    'order_count': order_count, 'has_attachment': has_attachment,
    'priority_flag': priority_flag, 'sentiment_score': sentiment,
    'category': category,
})

cat_names = {0: 'billing', 1: 'delivery', 2: 'product_quality', 3: 'technical_support', 4: 'account'}
print(f"Dataset shape: {df.shape}")
print(f"Category distribution:\n{df['category'].map(cat_names).value_counts()}")
df.head()

Dataset shape: (5000, 9)
Category distribution:
category
billing              3928
delivery              856
technical_support     149
product_quality        49
account                18
Name: count, dtype: int64


,word_count,urgency_score,response_time_hrs,customer_tenure_months,order_count,has_attachment,priority_flag,sentiment_score,category
0,44,1.68,9.0,10,4,0,1,-0.469,0
1,30,1.90,4.7,10,13,1,0,-0.865,1
2,48,4.61,47.0,18,7,1,0,-0.367,0
3,82,2.86,12.4,11,9,0,0,-0.306,0
4,28,2.47,12.2,1,8,0,0,0.579,1


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

DATA VALIDATION

Shape: (5000, 9)

Missing values:
Series([], dtype: int64)
No missing values.

Duplicate rows: 0

Target distribution:
category
0    3928
1     856
3     149
2      49
4      18
Name: count, dtype: int64

Dtypes:
word_count                  int64
urgency_score             float64
response_time_hrs         float64
customer_tenure_months      int64
order_count                 int32
has_attachment              int64
priority_flag               int64
sentiment_score           float64
category                    int64
dtype: object

Target 'category' confirmed.


## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, col in enumerate(num_cols[:8]):
    ax = axes[i // 4, i % 4]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col, fontsize=9)
plt.suptitle("Features by Complaint Category", fontsize=14)
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

corr = df[num_cols + [TARGET]].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Feature Correlations")
plt.tight_layout()
plt.show()
print(f"Features: {len(num_cols)}")

Features: 8


## 14 · Target Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title("Complaint Category Distribution")
cat_names = {0: 'billing', 1: 'delivery', 2: 'product', 3: 'technical', 4: 'account'}
axes[0].set_xticklabels([cat_names[i] for i in sorted(df[TARGET].unique())], rotation=45)
df[TARGET].value_counts().sort_index().plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title("Class Proportions"); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()
print(f"Class distribution:\n{df[TARGET].value_counts().sort_index()}")

Class distribution:
category
0    3928
1     856
2      49
3     149
4      18
Name: count, dtype: int64


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().sort_index().to_dict()}")

Train: (4000, 8), Test: (1000, 8)
Train target dist: {0: 3142, 1: 685, 2: 39, 3: 119, 4: 15}


## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [9]:
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean; X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (8): ['word_count', 'urgency_score', 'response_time_hrs', 'customer_tenure_months', 'order_count', 'has_attachment', 'priority_flag', 'sentiment_score']


## 18 · Baseline Model

Logistic Regression as baseline.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
n_classes = len(np.unique(y_train))
if n_classes == 2:
    y_prob_bl = baseline.predict_proba(X_test)[:, 1]
else:
    y_prob_bl = None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

BASELINE: Logistic Regression
Accuracy : 0.8730
Precision: 0.8375
Recall   : 0.8730
F1       : 0.8547


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision  Recall  Time Taken
Model                                                                                                        
NearestCentroid                   0.618           0.503234  0.913083  0.714168   0.870527   0.618    0.030979
CatBoostClassifier                0.862           0.395573  0.929535  0.845937   0.838978   0.862    7.766576
LinearDiscriminantAnalysis        0.869           0.390027  0.946869  0.849593   0.831625   0.869    0.021872
BaggingClassifier                 0.845           0.375690  0.899570  0.826358   0.818042   0.845    0.127432
DecisionTreeClassifier            0.795           0.367550  0.750600  0.798826   0.802861   0.795    0.032374
SGDClassifier                     0.852           0.363744       NaN  0.825629   0.819357   0.852    0.057977
PassiveAggressiveClassifier       0.848           0.362351       NaN  0.838248   0.830

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric="macro_f1",
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best: lgbm
Accuracy : 0.8470
F1       : 0.8328


## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [13]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test).ravel()
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost  Acc: 0.8710  F1: 0.8551


LightGBM  Acc: 0.8610  F1: 0.8437


XGBoost   Acc: 0.8570  F1: 0.8413


## 22 · Top 3 Model Selection

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
CatBoost                0.871  0.8551     0.8510   0.871
Logistic Regression     0.873  0.8547     0.8375   0.873
LightGBM                0.861  0.8437     0.8270   0.861
XGBoost                 0.857  0.8413     0.8534   0.857
FLAML                   0.847  0.8328     0.8217   0.847

Top 3: ['CatBoost', 'Logistic Regression', 'LightGBM']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")
plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))


Detailed Classification Reports — Top 3:

  CatBoost
              precision    recall  f1-score   support

           0       0.91      0.96      0.94       786
           1       0.67      0.67      0.67       171
           2       0.00      0.00      0.00        10
           3       0.50      0.07      0.12        30
           4       1.00      0.33      0.50         3

    accuracy                           0.87      1000
   macro avg       0.62      0.41      0.44      1000
weighted avg       0.85      0.87      0.86      1000


  Logistic Regression
              precision    recall  f1-score   support

           0       0.91      0.96      0.94       786
           1       0.69      0.68      0.69       171
           2       0.00      0.00      0.00        10
           3       0.00      0.00      0.00        30
           4       0.50      0.33      0.40         3

    accuracy                           0.87      1000
   macro avg       0.42      0.40      0.40      1000


## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]
errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

Best model: CatBoost
Error rate: 0.1290 (129 / 1000)

Errors by true class:
      errors  total  error_rate
true                           
0         32    786    0.040712
1         57    171    0.333333
2         10     10    1.000000
3         28     30    0.933333
4          2      3    0.666667


## 25 · Interpretation and Business Insight

- **Urgency score** and **word count** are strong indicators of billing complaints.
- **Response time** and **order count** predict delivery-related complaints.
- **Sentiment score** (negative) indicates product quality issues.
- **Priority flag** and urgency together signal technical support needs.
- **Customer tenure** helps distinguish account-related complaints.

## 26 · Limitations

1. Synthetic numeric features — real complaints use text (NLP needed).
2. No actual complaint text — features are pre-extracted.
3. Category boundaries are artificially clean.
4. 5 categories may not capture all real complaint types.
5. No temporal patterns (complaint volume spikes).

## 27 · How to Improve This Project

1. Use actual complaint text with NLP (TF-IDF, ModernBERT).
2. Add product category and customer segment features.
3. Include historical complaint resolution data.
4. Build a hierarchical classification (category → subcategory).
5. Add a 'miscellaneous' category for edge cases.

## 28 · Production Considerations

- Real-time ticket routing in customer support systems.
- Integration with CRM and helpdesk software.
- Confidence thresholds for manual review of uncertain tickets.
- Regular retraining as product and issue patterns evolve.

## 29 · Common Mistakes

1. Not using actual text features (the most informative signal).
2. Treating categories as equally important (some are more costly).
3. Not evaluating per-category performance.
4. Ignoring confidence scores for uncertain predictions.
5. Not handling the 'other' category for out-of-distribution tickets.

## 30 · Mini Challenge / Exercises

1. Which category has the highest error rate? Why?
2. Remove sentiment_score — which category suffers most?
3. Add a 'complaint_length_category' feature (short/medium/long).
4. Build a binary model for just 'billing vs everything else'.

## 31 · Final Summary / Key Takeaways

1. Customer complaint routing is a high-value business automation.
2. Numeric features provide moderate signal for categorization.
3. Real systems need NLP on actual complaint text.
4. Per-category evaluation reveals which types are hardest to route.
5. Confidence-based routing ensures uncertain tickets get human review.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }
with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))

Metrics saved.
{
  "CatBoost": {
    "accuracy": 0.871,
    "f1": 0.8551,
    "precision": 0.851,
    "recall": 0.871
  },
  "LightGBM": {
    "accuracy": 0.861,
    "f1": 0.8437,
    "precision": 0.827,
    "recall": 0.861
  },
  "XGBoost": {
    "accuracy": 0.857,
    "f1": 0.8413,
    "precision": 0.8534,
    "recall": 0.857
  },
  "Logistic Regression": {
    "accuracy": 0.873,
    "f1": 0.8547,
    "precision": 0.8375,
    "recall": 0.873
  },
  "FLAML": {
    "accuracy": 0.847,
    "f1": 0.8328,
    "precision": 0.8217,
    "recall": 0.847
  }
}
